In [2]:
# ================================================================
# Segmentation & CLTV View – 2D Grid / Heatmap
# CLTV_band (rows) vs Lead_score_band (cols) for customers
# ================================================================

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------
# 1. CONFIGURATION
# -----------------------

# Files (relative to notebooks/ directory)
CLTV_PATH   = Path("../data/processed/cltv_regression_scores.csv")
SCORES_PATH = Path("../data/processed/unified_customer_scores.csv")

# Common key column (exists in both)
KEY_COL  = "cust_id"                 # from your printout
CLTV_COL = "cltv_profit_ml"          # from cltv_regression_scores.csv
SCORE_COL = "ml_unified_customer_proba"  # from unified_customer_scores.csv

# Banding configuration
N_CLTV_BANDS  = 3   # 3 = Low / Mid / High
N_SCORE_BANDS = 3   # same for score bands

# Output directory (inside notebooks/)
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print("CLTV file    :", CLTV_PATH.resolve())
print("Scores file  :", SCORES_PATH.resolve())
print("Saving plots :", REPORT_DIR.resolve())

# -----------------------
# 2. LOAD DATA
# -----------------------

if not CLTV_PATH.exists():
    raise FileNotFoundError(f"CLTV file not found: {CLTV_PATH}")
if not SCORES_PATH.exists():
    raise FileNotFoundError(f"Scores file not found: {SCORES_PATH}")

df_cltv   = pd.read_csv(CLTV_PATH)
df_scores = pd.read_csv(SCORES_PATH)

print("\nCLTV columns  :", list(df_cltv.columns))
print("SCORES columns:", list(df_scores.columns))

# Check required columns
for col, path in [(KEY_COL, CLTV_PATH), (CLTV_COL, CLTV_PATH)]:
    if col not in df_cltv.columns:
        raise ValueError(f"Column '{col}' not found in {path.name}")
for col, path in [(KEY_COL, SCORES_PATH), (SCORE_COL, SCORES_PATH)]:
    if col not in df_scores.columns:
        raise ValueError(f"Column '{col}' not found in {path.name}")

# Deduplicate by key (safety)
df_cltv   = df_cltv.drop_duplicates(subset=[KEY_COL])
df_scores = df_scores.drop_duplicates(subset=[KEY_COL])

# -----------------------
# 3. MERGE CLTV + SCORE
# -----------------------

df = pd.merge(
    df_cltv[[KEY_COL, CLTV_COL]],
    df_scores[[KEY_COL, SCORE_COL]],
    on=KEY_COL,
    how="inner",
)

print("\nMerged rows:", len(df))

# Drop missing values
df = df.dropna(subset=[CLTV_COL, SCORE_COL]).copy()
print("After dropping NA:", len(df))

# -----------------------
# 4. CREATE BANDS (SEGMENTS)
# -----------------------

# CLTV bands by quantiles
df["cltv_band"] = pd.qcut(
    df[CLTV_COL],
    q=N_CLTV_BANDS,
    labels=["CLTV_Low", "CLTV_Mid", "CLTV_High"],
)

# Score bands by quantiles
df["score_band"] = pd.qcut(
    df[SCORE_COL],
    q=N_SCORE_BANDS,
    labels=["Score_Low", "Score_Mid", "Score_High"],
)

print("\nCLTV_band distribution:")
print(df["cltv_band"].value_counts().sort_index())
print("\nScore_band distribution:")
print(df["score_band"].value_counts().sort_index())

# -----------------------
# 5. BUILD 2D GRID (PIVOT)
# -----------------------

pivot_counts = pd.pivot_table(
    df,
    index="cltv_band",
    columns="score_band",
    values=KEY_COL,
    aggfunc="count",
    fill_value=0,
)

print("\nPivot – counts:")
print(pivot_counts)

# -----------------------
# 6. PLOT HEATMAP (COUNTS)
# -----------------------

plt.rcParams["figure.figsize"] = (6, 5)
plt.rcParams["figure.dpi"] = 120

fig, ax = plt.subplots()

data = pivot_counts.values
im = ax.imshow(data)

# Ticks and labels
ax.set_xticks(range(pivot_counts.shape[1]))
ax.set_yticks(range(pivot_counts.shape[0]))
ax.set_xticklabels(pivot_counts.columns)
ax.set_yticklabels(pivot_counts.index)

plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

# Add counts as text
for i in range(pivot_counts.shape[0]):
    for j in range(pivot_counts.shape[1]):
        ax.text(
            j, i,
            str(data[i, j]),
            ha="center", va="center",
        )

ax.set_xlabel("Lead Score Band")
ax.set_ylabel("CLTV Band")
ax.set_title("Customer Segmentation – CLTV vs Lead Score (Counts)")

fig.colorbar(im, ax=ax, label="Customer count")

plt.tight_layout()
heatmap_path = REPORT_DIR / "cltv_score_segmentation_heatmap.png"
plt.savefig(heatmap_path, bbox_inches="tight")
plt.close()

print("\nSaved heatmap to:", heatmap_path)



CLTV file    : /Users/mayanksuryavanshi/bank_lead_scoring_project/data/processed/cltv_regression_scores.csv
Scores file  : /Users/mayanksuryavanshi/bank_lead_scoring_project/data/processed/unified_customer_scores.csv
Saving plots : /Users/mayanksuryavanshi/bank_lead_scoring_project/notebooks/reports

CLTV columns  : ['cust_id', 'cltv_profit_ml']
SCORES columns: ['cust_id', 'ml_unified_customer_proba']

Merged rows: 1000
After dropping NA: 1000

CLTV_band distribution:
cltv_band
CLTV_Low     333
CLTV_Mid     333
CLTV_High    334
Name: count, dtype: int64

Score_band distribution:
score_band
Score_Low     334
Score_Mid     332
Score_High    334
Name: count, dtype: int64

Pivot – counts:
score_band  Score_Low  Score_Mid  Score_High
cltv_band                                   
CLTV_Low          111        114         108
CLTV_Mid          105        125         103
CLTV_High         118         93         123


/var/folders/ly/yq2pldk94bv2mtwrx1cf5dj80000gn/T/ipykernel_55583/3162445340.py:108: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_counts = pd.pivot_table(



Saved heatmap to: reports/cltv_score_segmentation_heatmap.png
